<a href="https://colab.research.google.com/github/mak-shah/COO/blob/master/Finetuning_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab's free-tier runtime already ships a GPU-matched PyTorch build — reinstalling it
# here can pull a CUDA build that doesn't match the preinstalled driver, and wastes several
# minutes + disk downloading torch's ~2GB wheel for nothing. We leave torch alone and only
# pin the small post-training libraries.
#
# Versions are pinned (rather than `-U`, i.e. "always latest") because this stack moves fast:
# transformers crossed the v5.0 major-version boundary and trl crossed v1.0 in 2026, and both
# now ship weekly/frequent releases that rename or remove TrainingArguments/SFTConfig fields
# (this notebook was breaking on `warmup_ratio` and `logging_dir`, both removed in transformers
# v5 — see the note near the training configuration cell below). The versions below are a
# combination that has been verified to work end-to-end for this notebook.
!pip install -q "transformers==5.15.0" "trl==1.10.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0" "tensorboard==2.21.0" "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.21.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 

In [ ]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


In [ ]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

def print_trainable_params(model):
    """Display trainable vs. total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Decode only the newly generated tokens
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [ ]:
# ============================================================
# Configuration — change these to experiment
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B"      # Base model (no -Instruct suffix!)
DATASET_NAME = "yahma/alpaca-cleaned"   # Classic instruction-tuning dataset
NUM_TRAIN_SAMPLES = 2000               # Subset for fast training on Colab

#Maximum number of tokens a Model can process in a single input sequence
MAX_SEQ_LENGTH = 512                   # Maximum sequence length

# LoRA hyperparameters
LORA_R = 16               # Rank
LORA_ALPHA = 32            # Scaling factor (2x rank)
LORA_DROPOUT = 0.05        # Regularization
LORA_TARGET_MODULES = [    # Apply to all linear layers
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training hyperparameters
LEARNING_RATE = 2e-4       # 10x higher than full fine-tuning
NUM_EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4       # Effective batch size = 4 * 4 = 16
WARMUP_RATIO = 0.03
OUTPUT_DIR = "./sft-lora-qwen"
LOG_DIR = f"{OUTPUT_DIR}/runs"  # TensorBoard log directory

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure pad token is set (base models often lack one)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Model max length: {tokenizer.model_max_length:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vocabulary size: 151,665
Model max length: 131,072
Pad token: '<|endoftext|>' (id=151643)


In [ ]:
# Load base model in float16 (standard LoRA — not quantized)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,  # `dtype` replaces the deprecated `torch_dtype` arg in transformers v5
    device_map="auto",
    trust_remote_code=True,
)

total_params = sum(p.numel() for p in model.parameters())
model_size_gb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3
print(f"\nBase model loaded: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Model size in memory: {model_size_gb:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Base model loaded: Qwen/Qwen2.5-1.5B
Total parameters: 1,543,714,304
Model size in memory: 2.88 GB
GPU memory allocated: 2.88 GB


In [ ]:
# Test prompts to evaluate instruction-following ability
test_prompts = [
    "Explain the concept of recursion in programming in 2-3 sentences.",
    "Write a Python function that checks if a number is prime.",
    "What are three benefits of regular exercise?",
]

print("=" * 70)
print("BASE MODEL RESPONSES (before SFT)")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'─' * 60}")
    response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
    print(f"Response: {response[:500]}")

# Store base responses for later comparison
base_responses = []
for prompt in test_prompts:
    base_responses.append(generate_response(model, tokenizer, prompt, max_new_tokens=200))

BASE MODEL RESPONSES (before SFT)

────────────────────────────────────────────────────────────
Prompt 1: Explain the concept of recursion in programming in 2-3 sentences.
────────────────────────────────────────────────────────────
Response: ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstream
ostringstr

────────────────────────────────────────────────────────────
Prompt 2: Write a Python function that checks if a number is prime.
────────────────────────────────────────────────────────────
Response: def is_prime(n):
    if n <= 1:
        return False
  

In [ ]:
# Load dataset
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset size: {len(raw_dataset):,} examples")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nExample entry:")
print(raw_dataset[0])

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Full dataset size: 51,760 examples
Columns: ['output', 'input', 'instruction']

Example entry:
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [ ]:
def format_alpaca_to_chat(example):
    """
    Convert Alpaca format (instruction/input/output) to conversational
    messages format for SFTTrainer.

    Alpaca format:
        {"instruction": "...", "input": "...", "output": "..."}

    Target format (ChatML-style messages):
        {"messages": [
            {"role": "system", "content": "..."},
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]}
    """
    # Combine instruction and input (if present) into user message
    if example.get("input") and example["input"].strip():
        user_content = f"{example['instruction']}\n\nInput: {example['input']}"
    else:
        user_content = example["instruction"]

    return {
        "messages": [
            {"role": "system", "content": "You are a helpful, accurate, and concise assistant."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["output"]},
        ]
    }

# Apply formatting and take a subset for training
dataset = raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))
dataset = dataset.map(
    format_alpaca_to_chat,
    remove_columns=raw_dataset.column_names,
    desc="Formatting to chat messages",
)

print(f"\nFormatted dataset size: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print(f"\nExample formatted entry:")
print(dataset[0])

Formatting to chat messages:   0%|          | 0/2000 [00:00<?, ? examples/s]


Formatted dataset size: 2,000
Columns: ['messages']

Example formatted entry:
{'messages': [{'role': 'system', 'content': 'You are a helpful, accurate, and concise assistant.'}, {'role': 'user', 'content': 'Rearrange the following sentence to make the sentence more interesting.\n\nInput: She left the party early'}, {'role': 'assistant', 'content': 'Early, she left the party.'}]}


In [ ]:
# Verify the chat template renders correctly
sample = dataset[0]
rendered = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
print("Rendered chat template (first example):\n")
print(rendered[:800])

Rendered chat template (first example):

<|im_start|>system
You are a helpful, accurate, and concise assistant.<|im_end|>
<|im_start|>user
Rearrange the following sentence to make the sentence more interesting.

Input: She left the party early<|im_end|>
<|im_start|>assistant
Early, she left the party.<|im_end|>



In [ ]:
# Define LoRA configuration
peft_config = LoraConfig(
    r=LORA_R,                           # Rank of decomposition
    lora_alpha=LORA_ALPHA,              # Scaling factor (alpha/r applied)
    lora_dropout=LORA_DROPOUT,          # Dropout for regularization
    bias="none",                        # Don't train bias terms
    task_type="CAUSAL_LM",             # Decoder-only language model
    target_modules=LORA_TARGET_MODULES, # All attention + MLP layers
)

print("LoRA Configuration:")
print(f"  Rank (r):          {peft_config.r}")
print(f"  Alpha (α):         {peft_config.lora_alpha}")
print(f"  Effective scaling: {peft_config.lora_alpha / peft_config.r}")
print(f"  Dropout:           {peft_config.lora_dropout}")
print(f"  Target modules:    {peft_config.target_modules}")
print(f"  Task type:         {peft_config.task_type}")

LoRA Configuration:
  Rank (r):          16
  Alpha (α):         32
  Effective scaling: 2.0
  Dropout:           0.05
  Target modules:    {'down_proj', 'k_proj', 'v_proj', 'o_proj', 'q_proj', 'up_proj', 'gate_proj'}
  Task type:         CAUSAL_LM


In [ ]:
# Apply LoRA to the model and inspect
lora_model = get_peft_model(model, peft_config)

print("\n" + "=" * 50)
print("PARAMETER COMPARISON")
print("=" * 50)
print_trainable_params(lora_model)

# Show the LoRA module structure for one layer
print("\n\nLoRA module example (layer 0, q_proj):")
print(lora_model.model.model.layers[0].self_attn.q_proj)


PARAMETER COMPARISON
Trainable parameters:   18,464,768  (1.1820%)
Total parameters:     1,562,179,072


LoRA module example (layer 0, q_proj):
lora.Linear(
  (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=1536, out_features=16, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=16, out_features=1536, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


In [ ]:
# Manual calculation of LoRA parameters
# For each target module, LoRA adds: r * (d_in + d_out) parameters
# (A is r x d_in, B is d_out x r)
print("Manual LoRA Parameter Calculation:")
print("-" * 50)

total_lora_params = 0
sample_layer = lora_model.model.model.layers[0]

# Check dimensions of each target module type
for name in LORA_TARGET_MODULES:
    # Navigate to the module
    if "proj" in name and ("q_" in name or "k_" in name or "v_" in name or "o_" in name):
        module = getattr(sample_layer.self_attn, name)
    else:
        module = getattr(sample_layer.mlp, name)

    if hasattr(module, 'lora_A'):
        # Get dimensions
        A_shape = module.lora_A['default'].weight.shape  # (r, in_features)
        B_shape = module.lora_B['default'].weight.shape  # (out_features, r)
        params_per_layer = A_shape[0] * A_shape[1] + B_shape[0] * B_shape[1]
        print(f"  {name:12s}: A={list(A_shape)}, B={list(B_shape)} → {params_per_layer:,} params/layer")
        total_lora_params += params_per_layer

num_layers = len(lora_model.model.model.layers)
print(f"\nParameters per transformer block: {total_lora_params:,}")
print(f"Number of transformer blocks:     {num_layers}")
print(f"Total LoRA parameters:            {total_lora_params * num_layers:,}")

Manual LoRA Parameter Calculation:
--------------------------------------------------
  q_proj      : A=[16, 1536], B=[1536, 16] → 49,152 params/layer
  k_proj      : A=[16, 1536], B=[256, 16] → 28,672 params/layer
  v_proj      : A=[16, 1536], B=[256, 16] → 28,672 params/layer
  o_proj      : A=[16, 1536], B=[1536, 16] → 49,152 params/layer
  gate_proj   : A=[16, 1536], B=[8960, 16] → 167,936 params/layer
  up_proj     : A=[16, 1536], B=[8960, 16] → 167,936 params/layer
  down_proj   : A=[16, 8960], B=[1536, 16] → 167,936 params/layer

Parameters per transformer block: 659,456
Number of transformer blocks:     28
Total LoRA parameters:            18,464,768


In [ ]:
# Remove the PEFT wrapper for now — SFTTrainer will re-apply it via peft_config
# (SFTTrainer expects either a base model + peft_config, or a pre-wrapped PEFT model)
del lora_model
clear_memory()

# Reload the base model fresh for SFTTrainer
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,  # `dtype` replaces the deprecated `torch_dtype` arg in transformers v5
    device_map="auto",
    trust_remote_code=True,
)

GPU allocated: 2.95 GB
GPU reserved:  2.99 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Training configuration
#
# NOTE (transformers v5 fix): `warmup_ratio` and `logging_dir` were both removed from
# `TrainingArguments` (which `SFTConfig` extends) in transformers v5. `warmup_ratio` is replaced
# by `warmup_steps`, which now accepts either a float < 1 (treated as a ratio of total steps, same
# behavior as the old `warmup_ratio`) or an int >= 1 (treated as an absolute step count). `logging_dir`
# is replaced by the `TENSORBOARD_LOGGING_DIR` environment variable, which the TensorBoard callback
# reads when it starts up — so we set it right before building the config.
os.environ["TENSORBOARD_LOGGING_DIR"] = LOG_DIR

training_args = SFTConfig(
    # ── Output ──────────────────────────────────────────────
    output_dir=OUTPUT_DIR,

    # ── Core training hyperparameters ───────────────────────
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,           # 2e-4: the ~10x LoRA multiplier
    weight_decay=0.01,
    max_grad_norm=1.0, #To prevent exploding gradients
    warmup_steps=WARMUP_RATIO,             # float < 1 == ratio of total steps (replaces warmup_ratio)
    lr_scheduler_type="cosine",

    # ── Precision ───────────────────────────────────────────
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    # ── Memory optimization ─────────────────────────────────
    gradient_checkpointing=True,           # Trade compute for memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",

    # ── SFT-specific settings ───────────────────────────────
    max_length=MAX_SEQ_LENGTH,
    packing=False,                         # Set True for efficiency with short examples
    dataset_kwargs={"skip_prepare_dataset": False},

    # ── Logging to TensorBoard ──────────────────────────────
    # (logging_dir is set via the TENSORBOARD_LOGGING_DIR env var above, not a kwarg here)
    logging_steps=10,
    logging_first_step=True,
    report_to="tensorboard",               # Enable TensorBoard logging

    # ── Saving ──────────────────────────────────────────────
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # ── Reproducibility ─────────────────────────────────────
    seed=42,
)

print("Training Configuration Summary:")
print(f"  Effective batch size:  {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Learning rate:         {LEARNING_RATE}")
print(f"  Epochs:                {NUM_EPOCHS}")
print(f"  Max sequence length:   {MAX_SEQ_LENGTH}")
print(f"  Precision:             {'bf16' if training_args.bf16 else 'fp16'}")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"  TensorBoard log dir:   {LOG_DIR}")
print(f"  Logging every:         {training_args.logging_steps} steps")

Training Configuration Summary:
  Effective batch size:  16
  Learning rate:         0.0002
  Epochs:                1
  Max sequence length:   512
  Precision:             bf16
  Gradient checkpointing: True
  TensorBoard log dir:   ./sft-lora-qwen/runs
  Logging every:         10 steps


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    #Comment this out to do full finetuning
    peft_config=peft_config,             # LoRA is applied here automatically
)

# Show the effective model
print("Model with LoRA adapters applied:")
print_trainable_params(trainer.model)

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Model with LoRA adapters applied:
Trainable parameters:   18,464,768  (1.1820%)
Total parameters:     1,562,179,072


In [ ]:
# ════════════════════════════════════════════════
# TRAIN!
# ════════════════════════════════════════════════
print("\nStarting SFT with LoRA...")
print(f"Training on {len(dataset):,} examples for {NUM_EPOCHS} epoch(s)")
print(f"Metrics logging to TensorBoard every {training_args.logging_steps} steps\n")

train_result = trainer.train()

# Print training metrics
print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
metrics = train_result.metrics
print(f"  Total training time: {metrics.get('train_runtime', 0):.1f} seconds")
print(f"  Samples/second:      {metrics.get('train_samples_per_second', 0):.2f}")
print(f"  Final loss:          {metrics.get('train_loss', 0):.4f}")
print(f"  Total steps:         {metrics.get('total_flos', 0):.2e} FLOPs")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting SFT with LoRA...
Training on 2,000 examples for 1 epoch(s)
Metrics logging to TensorBoard every 10 steps



Step,Training Loss
1,2.242395
10,1.858914
20,1.359848
30,1.311827
40,1.363624
50,1.296884
60,1.277460
70,1.301445


Step,Training Loss
1,2.242395
10,1.858914
20,1.359848
30,1.311827
40,1.363624
50,1.296884
60,1.277460
70,1.301445
80,1.400614
90,1.222179



TRAINING COMPLETE
  Total training time: 3037.1 seconds
  Samples/second:      0.66
  Final loss:          1.3549
  Total steps:         5.17e+15 FLOPs


In [ ]:
# Save the trained LoRA adapter (NOT the full model)
adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Check adapter size
adapter_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"\nAdapter saved to: {adapter_path}")
print(f"Adapter size: {adapter_size / 1024**2:.1f} MB")
print(f"(Compare to full model: ~{sum(p.numel() * 2 for p in model.parameters()) / 1024**3:.1f} GB)")


Adapter saved to: ./sft-lora-qwen/final-adapter
Adapter size: 81.4 MB
(Compare to full model: ~2.9 GB)


In [ ]:
# The trainer.model already has the LoRA weights — use it directly
finetuned_model = trainer.model
finetuned_model.eval()

print("=" * 70)
print("FINE-TUNED MODEL RESPONSES (after SFT with LoRA)")
print("=" * 70)

finetuned_responses = []
for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'─' * 60}")
    response = generate_response(finetuned_model, tokenizer, prompt, max_new_tokens=200)
    finetuned_responses.append(response)
    print(f"Response: {response[:500]}")

In [ ]:
print("=" * 80)
print(" COMPARISON: BASE MODEL vs. FINE-TUNED MODEL (LoRA SFT)")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'━' * 80}")
    print(f"  PROMPT {i+1}: {prompt}")
    print(f"{'━' * 80}")

    print(f"\n  🔴 BASE MODEL:")
    print(f"  {base_responses[i][:300]}")

    print(f"\n  🟢 FINE-TUNED (LoRA SFT):")
    print(f"  {finetuned_responses[i][:300]}")
    print()

In [ ]:
# ── Method 1: Load base model + LoRA adapter separately ─────
from peft import PeftModel

# Load fresh base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,  # `dtype` replaces the deprecated `torch_dtype` arg in transformers v5
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Attach the saved LoRA adapter
model_with_adapter = PeftModel.from_pretrained(base_model, adapter_path)
model_with_adapter.eval()

# Test it
response = generate_response(model_with_adapter, tokenizer, "What is machine learning?", max_new_tokens=150)
print("Response from loaded adapter:")
print(response)

In [ ]:
# ── Method 2: Merge adapter into base weights (for deployment) ──
# After merging, the model is a standard transformer with no adapter overhead
merged_model = model_with_adapter.merge_and_unload()

print(f"Merged model type: {type(merged_model).__name__}")
print(f"Total parameters:  {sum(p.numel() for p in merged_model.parameters()):,}")

# The merged model behaves identically but has no PEFT dependency
response = generate_response(merged_model, tokenizer, "What is machine learning?", max_new_tokens=150)
print("\nResponse from merged model:")
print(response)

# Save the full merged model (larger, but no PEFT dependency at inference)
# merged_model.save_pretrained("./merged-model")
# tokenizer.save_pretrained("./merged-model")